<a href="https://colab.research.google.com/github/hemanthgm914856/CrewAI-Smart-Schedule-Manager/blob/main/CrewAI_Smart_Schedule_Manager.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip -q install -U crewai google-genai google-api-python-client google-auth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 9.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.4/232.4 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.0/268.0 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/

In [4]:
!pip install -q -U google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client

In [5]:
import os
import getpass
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

In [6]:
from google.colab import auth

auth.authenticate_user()

In [7]:
from google.colab import files

uploaded = files.upload()
credentials_file = next(iter(uploaded))

Saving credentials_calander.json to credentials_calander.json


In [8]:
from google_auth_oauthlib.flow import Flow

SCOPES = [
    "https://www.googleapis.com/auth/calendar.readonly"
]

flow = Flow.from_client_secrets_file(
    credentials_file,
    scopes=SCOPES,
    redirect_uri="http://localhost:8080/"
)

auth_url, _ = flow.authorization_url(
    access_type="offline",
    prompt="consent"
)

print("Open this URL and authorize access:\n")
print(auth_url)

Open this URL and authorize access:

https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=235856800676-r1kjk7ngo6jravh5e28acgpuljbh1i1l.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcalendar.readonly&state=nD1gYcwD4XOIYLKhK58DHjVRhcymnD&code_challenge=wENYml1gQNmX6A5PJghJvZ9Jod8bovsfxqDm64jK65I&code_challenge_method=S256&access_type=offline&prompt=consent


In [9]:
import os
from googleapiclient.discovery import build

os.environ["OAUTHLIB_INSECURE_TRANSPORT"] = "1"

redirected_url = input(
    "Paste the complete redirected URL: "
).strip()

flow.fetch_token(
    authorization_response=redirected_url
)

credentials = flow.credentials

service = build(
    "calendar",
    "v3",
    credentials=credentials
)

calendar = service.calendarList().get(
    calendarId="primary"
).execute()

print("Google Calendar authentication successful.")
print("Connected calendar:", calendar.get("summary"))

Paste the complete redirected URL: http://localhost:8080/?state=nD1gYcwD4XOIYLKhK58DHjVRhcymnD&iss=https://accounts.google.com&code=4/0AXlqoi5sk4KxK86W01ewHPwqiyemxk8FR2tZwCCQGGFIRWg0KWRrEbJmVDF-9r5_7His_A&scope=https://www.googleapis.com/auth/calendar.readonly
Google Calendar authentication successful.
Connected calendar: hemanthcitcoorg@gmail.com


In [16]:
from datetime import datetime, time, timedelta
from zoneinfo import ZoneInfo

IST = ZoneInfo("Asia/Kolkata")
today = datetime.now(IST).date()

start = datetime.combine(today, time.min, tzinfo=IST)
end = start + timedelta(days=1)

response = service.events().list(
    calendarId="primary",
    timeMin=start.isoformat(),
    timeMax=end.isoformat(),
    singleEvents=True,
    orderBy="startTime",
    maxResults=250
).execute()

events = response.get("items", [])

calendar_schedule = []

for event in events:
    calendar_schedule.append({
        "event": event.get("summary", "Untitled event"),
        "start": event["start"].get(
            "dateTime", event["start"].get("date")
        ),
        "end": event["end"].get(
            "dateTime", event["end"].get("date")
        )
    })

print("Today's Schedule")
print("=" * 30)

if calendar_schedule:
    for i, event in enumerate(calendar_schedule, 1):
        print(f"{i}. {event['event']}")
        print(f"   Start: {event['start']}")
        print(f"   End:   {event['end']}")
else:
    print("No events found for today.")

Today's Schedule
1. Last day of course
   Start: 2026-10-09T09:00:00Z
   End:   2026-10-09T10:00:00Z


In [17]:
calendar_list = service.calendarList().list().execute()

for calendar in calendar_list.get("items", []):
    print(
        "Calendar:", calendar["summary"],
        "| ID:", calendar["id"],
        "| Primary:", calendar.get("primary", False)
    )

Calendar: hemanthcitcoorg@gmail.com | ID: hemanthcitcoorg@gmail.com | Primary: True
Calendar: Holidays in India | ID: en.indian#holiday@group.v.calendar.google.com | Primary: False


In [18]:
from datetime import datetime, time, timedelta
from zoneinfo import ZoneInfo

IST = ZoneInfo("Asia/Kolkata")
today = datetime.now(IST).date()

start = datetime.combine(today, time.min, tzinfo=IST)
end = start + timedelta(days=1)

response = service.events().list(
    calendarId="primary",
    timeMin=start.isoformat(),
    timeMax=end.isoformat(),
    singleEvents=True,
    orderBy="startTime",
    maxResults=250
).execute()

events = response.get("items", [])
calendar_schedule = []

for event in events:
    event_start = event["start"].get(
        "dateTime", event["start"].get("date")
    )
    event_end = event["end"].get(
        "dateTime", event["end"].get("date")
    )

    if "T" in event_start:
        event_start = datetime.fromisoformat(
            event_start.replace("Z", "+00:00")
        ).astimezone(IST).strftime("%Y-%m-%d %I:%M %p")

        event_end = datetime.fromisoformat(
            event_end.replace("Z", "+00:00")
        ).astimezone(IST).strftime("%Y-%m-%d %I:%M %p")

    calendar_schedule.append({
        "event": event.get("summary", "Untitled event"),
        "start": event_start,
        "end": event_end
    })

print("Today's Schedule")
print("=" * 30)

if calendar_schedule:
    for i, event in enumerate(calendar_schedule, 1):
        print(f"{i}. {event['event']}")
        print(f"   Start: {event['start']}")
        print(f"   End:   {event['end']}")
else:
    print("No events found for today.")

Today's Schedule
1. Last day of course
   Start: 2026-10-09 02:30 PM
   End:   2026-10-09 03:30 PM


In [22]:
from crewai import Agent, Task, Crew, LLM
import os
import json

llm = LLM(
    model="gemini/gemini-3.5-flash",
    api_key=os.environ["GEMINI_API_KEY"]
)

alarm_agent = Agent(
    role="Alarm Setter",
    goal="Calculate accurate alarm times for calendar events",
    backstory="You calculate alarms five minutes before scheduled events.",
    llm=llm,
    verbose=True
)

reminder_agent = Agent(
    role="Schedule Reminder Manager",
    goal="Prepare reminders for upcoming calendar events",
    backstory="You prepare clear reminders ten minutes before events.",
    llm=llm,
    verbose=True
)

schedule_agent = Agent(
    role="Daily Schedule Viewer",
    goal="Display every event in the day's calendar",
    backstory="You organize calendar events chronologically.",
    llm=llm,
    verbose=True
)

print("Three CrewAI agents created successfully.")

Three CrewAI agents created successfully.


In [23]:
alarm_task = Task(
    description="""
    Using the original calendar schedule below, calculate an alarm time
    exactly 5 minutes before each timed event starts.
    Include the event name, event start time, and calculated alarm time.
    Do not invent events. Flag all-day events separately.

    Original schedule:
    {schedule}
    """,
    expected_output="A list of events with their start times and alarms 5 minutes before each event.",
    agent=alarm_agent
)

reminder_task = Task(
    description="""
    Using the original calendar schedule below, calculate a reminder time
    exactly 10 minutes before each timed event starts.
    Include the event name, event start time, and calculated reminder time.
    Do not invent events. Flag all-day events separately.

    Original schedule:
    {schedule}
    """,
    expected_output="A list of events with their start times and reminders 10 minutes before each event.",
    agent=reminder_agent
)

schedule_task = Task(
    description="""
    Display all events from the original calendar schedule in chronological
    order. Include each event's name, start time, and end time.
    Do not omit or invent events. Flag all-day events separately.

    Original schedule:
    {schedule}
    """,
    expected_output="A complete chronological list of the day's calendar events.",
    agent=schedule_agent
)

print("Three CrewAI tasks created successfully.")

Three CrewAI tasks created successfully.


In [24]:
crew = Crew(
    agents=[alarm_agent, reminder_agent, schedule_agent],
    tasks=[alarm_task, reminder_task, schedule_task],
    verbose=True
)

result = await crew.kickoff_async(
    inputs={
        "schedule": json.dumps(
            calendar_schedule,
            ensure_ascii=False,
            indent=2
        )
    }
)

print("\nCREWAI AGENT OUTPUTS")
print("=" * 60)
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 518149d0-9094-4334-8511-a32d3eba94d0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│      Using the original calendar schedule below, calculate an alarm time                                        │
│      exactly 5 minutes before each timed event starts.                                                          │
│      Include the event name, event start time, and calculated alarm time.                                       │
│      Do not invent events. Flag all-day events separately.                                                      │
│                                                                                                                 │
│      Original schedule:                                                                                         │
│      [                                                                                                          │
│    {                                                                                                            │
│      "event": "Last day of course",                                                                             │
│      "start": "2026-10-09 02:30 PM",                                                                            │
│      "end": "2026-10-09 03:30 PM"                                                                               │
│    }                                                                                                            │
│  ]                                                                                                              │
│                                                                                                                 │
│  ID: 727e8cac-8907-4cc5-9241-5f77126ce767                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Alarm Setter                                                                                            │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Using the original calendar schedule below, calculate an alarm time                                        │
│      exactly 5 minutes before each timed event starts.                                                          │
│      Include the event name, event start time, and calculated alarm time.                                       │
│      Do not invent events. Flag all-day events separately.                                                      │
│                                                                                                                 │
│      Original schedule:                                                                                         │
│      [                                                                                                          │
│    {                                                                                                            │
│      "event": "Last day of course",                                                                             │
│      "start": "2026-10-09 02:30 PM",                                                                            │
│      "end": "2026-10-09 03:30 PM"                                                                               │
│    }                                                                                                            │
│  ]                                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Alarm Setter                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Here is the calculated alarm time for your scheduled event:                                                    │
│                                                                                                                 │
│  *   **Event:** Last day of course                                                                              │
│      *   **Start Time:** 2026-10-09 02:30 PM                                                                    │
│      *   **Alarm Time (5 minutes before):** 2026-10-09 02:25 PM                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Daily Schedule Viewer                                                                                   │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Here is the complete chronological schedule of the day's calendar events based on the original schedule        │
│  provided:                                                                                                      │
│                                                                                                                 │
│  **Chronological Schedule (October 9, 2026):**                                                                  │
│                                                                                                                 │
│  1. **Last day of course**                                                                                      │
│     * **Start Time:** 2026-10-09 02:30 PM                                                                       │
│     * **End Time:** 2026-10-09 03:30 PM                                                                         │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **All-Day Events:**                                                                                            │
│  * None                                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│      Display all events from the original calendar schedule in chronological                                    │
│      order. Include each event's name, start time, and end time.                                                │
│      Do not omit or invent events. Flag all-day events separately.                                              │
│                                                                                                                 │
│      Original schedule:                                                                                         │
│      [                                                                                                          │
│    {                                                                                                            │
│      "event": "Last day of course",                                                                             │
│      "start": "2026-10-09 02:30 PM",                                                                            │
│      "end": "2026-10-09 03:30 PM"                                                                               │
│    }                                                                                                            │
│  ]                                                                                                              │
│                                                                                                                 │
│  Agent: Daily Schedule Viewer                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 518149d0-9094-4334-8511-a32d3eba94d0                                                                       │
│  Final Output: Here is the complete chronological schedule of the day's calendar events based on the original   │
│  schedule provided:                                                                                             │
│                                                                                                                 │
│  **Chronological Schedule (October 9, 2026):**                                                                  │
│                                                                                                                 │
│  1. **Last day of course**                                                                                      │
│     * **Start Time:** 2026-10-09 02:30 PM                                                                       │
│     * **End Time:** 2026-10-09 03:30 PM                                                                         │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **All-Day Events:**                                                                                            │
│  * None                                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


CREWAI AGENT OUTPUTS
Here is the complete chronological schedule of the day's calendar events based on the original schedule provided:

**Chronological Schedule (October 9, 2026):**

1. **Last day of course**
   * **Start Time:** 2026-10-09 02:30 PM
   * **End Time:** 2026-10-09 03:30 PM

---

**All-Day Events:**
* None


In [28]:
from google import genai

client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
)

agent_outputs = []

for i, task_output in enumerate(result.tasks_output, start=1):
    agent_outputs.append({
        "agent": f"Agent {i}",
        "output": str(task_output)
    })

verification_prompt = f"""
You are an independent Verification LLM for a Smart Schedule Manager.

ORIGINAL GOOGLE CALENDAR SCHEDULE:
{json.dumps(calendar_schedule, ensure_ascii=False, indent=2)}

OUTPUTS FROM THE THREE CREWAI AGENTS:
{json.dumps(agent_outputs, ensure_ascii=False, indent=2)}

Verify the outputs against the original schedule.

Rules:
1. Check that every original event is included.
2. Check that no event has been invented.
3. Verify each alarm is exactly 5 minutes before the event starts.
4. Verify each reminder is exactly 10 minutes before the event starts.
5. Verify the daily schedule contains correct start and end times and is chronological.
6. Identify missing events, incorrect times, duplicates, and inconsistencies.
7. Correct errors using the original schedule as the source of truth.
8. For all-day events, do not invent a start time for alarms or reminders.
9. If an agent's output is missing or unclear, say so rather than guessing.

Return a report with these sections:
A. Verification Results
B. Errors Detected
C. Corrected Alarm Schedule
D. Corrected Reminder Schedule
E. Final Daily Schedule
F. Overall Status: PASS or FAIL

Only mark PASS if all outputs are consistent with the original schedule.
"""

verification_response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=verification_prompt
)

print("VERIFICATION LLM REPORT")
print("=" * 60)
print(verification_response.text)

VERIFICATION LLM REPORT
### A. Verification Results

*   **Agent 1 (Alarm Agent):** **PASSED.** The agent correctly identified the single scheduled event ("Last day of course"), verified its start time (2026-10-09 02:30 PM), and accurately calculated the alarm time to be exactly 5 minutes before the start time (2026-10-09 02:25 PM).
*   **Agent 2 (Reminder Agent):** **PASSED.** The agent correctly identified the scheduled event, verified its start time (2026-10-09 02:30 PM), and accurately calculated the reminder time to be exactly 10 minutes before the start time (2026-10-09 02:20 PM).
*   **Agent 3 (Chronological Schedule Agent):** **PASSED.** The agent compiled a complete, accurate chronological schedule containing the single timed event with correct start and end times. It also correctly identified that there were no all-day events.

---

### B. Errors Detected

*   **None.** All agent outputs are 100% accurate, consistent with the original schedule, and free of errors, missing inf

In [29]:
from datetime import datetime, timedelta

def parse_event_time(value):
    return datetime.strptime(value, "%Y-%m-%d %I:%M %p")

expected_alarms = {}
expected_reminders = {}
validation_errors = []

for event in calendar_schedule:
    start_text = event["start"]

    if "T" not in start_text and len(start_text) == 10:
        validation_errors.append(
            f"{event['event']}: All-day event; time-based alarm/reminder not calculated."
        )
        continue

    start_time = parse_event_time(start_text)

    expected_alarms[event["event"]] = (
        start_time - timedelta(minutes=5)
    ).strftime("%Y-%m-%d %I:%M %p")

    expected_reminders[event["event"]] = (
        start_time - timedelta(minutes=10)
    ).strftime("%Y-%m-%d %I:%M %p")

print("EXPECTED ALARMS")
for event, alarm in expected_alarms.items():
    print(f"{event}: {alarm}")

print("\nEXPECTED REMINDERS")
for event, reminder in expected_reminders.items():
    print(f"{event}: {reminder}")

print("\nPYTHON VALIDATION")
if expected_alarms and expected_reminders:
    print("Expected alarm and reminder times calculated independently.")
else:
    print("No timed events available to validate.")

for error in validation_errors:
    print(error)

EXPECTED ALARMS
Last day of course: 2026-10-09 02:25 PM

EXPECTED REMINDERS
Last day of course: 2026-10-09 02:20 PM

PYTHON VALIDATION
Expected alarm and reminder times calculated independently.


In [30]:
import json

final_report = {
    "project": "CrewAI Smart Schedule Manager",
    "original_calendar_schedule": calendar_schedule,
    "agent_outputs": agent_outputs,
    "verification_llm_report": verification_response.text,
    "python_expected_alarms": expected_alarms,
    "python_expected_reminders": expected_reminders,
    "python_validation_notes": validation_errors
}

print("=" * 70)
print("FINAL SMART SCHEDULE MANAGER REPORT")
print("=" * 70)

print("\n1. ORIGINAL CALENDAR EVENTS")
print(json.dumps(calendar_schedule, indent=2, ensure_ascii=False))

print("\n2. CREWAI AGENT OUTPUTS")
for item in agent_outputs:
    print(f"\n{item['agent']}")
    print(item["output"])

print("\n3. VERIFICATION LLM REPORT")
print(verification_response.text)

print("\n4. INDEPENDENT PYTHON CALCULATIONS")
print("\nExpected alarms:")
print(json.dumps(expected_alarms, indent=2, ensure_ascii=False))

print("\nExpected reminders:")
print(json.dumps(expected_reminders, indent=2, ensure_ascii=False))

print("\n5. PYTHON VALIDATION NOTES")
if validation_errors:
    for error in validation_errors:
        print("-", error)
else:
    print("No all-day-event warnings.")

with open("smart_schedule_final_report.json", "w", encoding="utf-8") as file:
    json.dump(final_report, file, indent=2, ensure_ascii=False)

print("\nFinal report saved as smart_schedule_final_report.json")

FINAL SMART SCHEDULE MANAGER REPORT

1. ORIGINAL CALENDAR EVENTS
[
  {
    "event": "Last day of course",
    "start": "2026-10-09 02:30 PM",
    "end": "2026-10-09 03:30 PM"
  }
]

2. CREWAI AGENT OUTPUTS

Agent 1
Here is the calculated alarm time for your scheduled event:

*   **Event:** Last day of course
    *   **Start Time:** 2026-10-09 02:30 PM
    *   **Alarm Time (5 minutes before):** 2026-10-09 02:25 PM

Agent 2
Based on the schedule provided, here is the calculated reminder time set exactly 10 minutes before the event starts:

*   **Event:** Last day of course
    *   **Start Time:** 2026-10-09 02:30 PM
    *   **Reminder Time (10 minutes before):** 2026-10-09 02:20 PM

Agent 3
Here is the complete chronological schedule of the day's calendar events based on the original schedule provided:

**Chronological Schedule (October 9, 2026):**

1. **Last day of course**
   * **Start Time:** 2026-10-09 02:30 PM
   * **End Time:** 2026-10-09 03:30 PM

---

**All-Day Events:**
* None



In [31]:
!pip -q install reportlab

from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import inch
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Preformatted, PageBreak
)
from reportlab.lib import colors
from xml.sax.saxutils import escape
import json
import os

pdf_filename = "Smart_Schedule_Manager_Report.pdf"

doc = SimpleDocTemplate(
    pdf_filename,
    pagesize=A4,
    rightMargin=0.6 * inch,
    leftMargin=0.6 * inch,
    topMargin=0.6 * inch,
    bottomMargin=0.6 * inch
)

styles = getSampleStyleSheet()
story = []

story.append(Paragraph("CrewAI Smart Schedule Manager", styles["Title"]))
story.append(Spacer(1, 8))
story.append(Paragraph("Final Project Report", styles["Heading2"]))
story.append(Spacer(1, 12))

def add_section(title, content):
    story.append(Paragraph(escape(title), styles["Heading2"]))
    story.append(Spacer(1, 6))
    formatted = json.dumps(content, indent=2, ensure_ascii=False) if not isinstance(content, str) else content
    story.append(
        Preformatted(
            escape(formatted),
            styles["Code"]
        )
    )
    story.append(Spacer(1, 12))

add_section("1. Original Google Calendar Schedule", calendar_schedule)
add_section("2. Alarm Setter Output", agent_outputs[0]["output"])
add_section("3. Reminder Manager Output", agent_outputs[1]["output"])
add_section("4. Daily Schedule Viewer Output", agent_outputs[2]["output"])
add_section("5. Verification LLM Report", verification_response.text)
add_section("6. Independently Calculated Alarm Times", expected_alarms)
add_section("7. Independently Calculated Reminder Times", expected_reminders)
add_section("8. Python Validation Notes", validation_errors or ["No all-day-event warnings."])

doc.build(story)

print(f"PDF report created: {os.path.abspath(pdf_filename)}")

from google.colab import files
files.download(pdf_filename)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 35.2 MB/s eta 0:00:00
PDF report created: /content/Smart_Schedule_Manager_Report.pdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>